# Task 4: PDF Data Extraction

The five invoices use five different layouts, so each layout has its own parser. Every invoice is checked twice: the line items must add up to the printed subtotal, and subtotal minus discount plus delivery plus tax must equal the printed total.

In [1]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd
import pdfplumber

here = Path.cwd()
PDF_DIR = next((p for p in [*[d / 'pdfs' for d in (here, *here.parents)], *here.glob('*/pdfs'), *here.glob('*/*/pdfs')]
                if p.is_dir() and any(p.glob('*.pdf'))), None)
if PDF_DIR is None:
    raise FileNotFoundError('Put the invoice PDFs in a folder called "pdfs" next to this notebook.')
OUT = PDF_DIR.parent
sorted(f.name for f in PDF_DIR.glob('*.pdf'))

['bluewave-invoice.pdf',
 'coral_tech-invoice.pdf',
 'hlp-invoice.pdf',
 'maison_vanille-invoice.pdf',
 'shadow-invoice.pdf']

## Helpers

In [2]:
def num(s):
    s = s.replace('$', '').replace('Rs', '').strip()
    if s in ('', '-'):
        return 0.0
    if ',' in s and '.' in s:
        s = s.replace(',', '')
    elif re.search(r',\d{2}$', s):
        s = s.replace(' ', '').replace('.', '').replace(',', '.')
    else:
        s = s.replace(',', '')
    return float(s.replace(' ', ''))


def iso(s, fmt='%d/%m/%Y'):
    return datetime.strptime(s.strip(), fmt).date().isoformat()


def find(pattern, text, cast=str, default=None):
    m = re.search(pattern, text, re.M)
    return cast(m.group(1)) if m else default


def section(text, start, end):
    lines = text.splitlines()
    i = next(n for n, l in enumerate(lines) if l.startswith(start))
    j = next(n for n, l in enumerate(lines) if n > i and l.startswith(end))
    return lines[i + 1:j]

## Parsers

In [3]:
def parse_bluewave(t):
    items = []
    for l in section(t, '# Description', 'Subtotal'):
        m = re.match(r'^(\d+) (.+) (\d+) ([\d,]+\.\d{2}) ([\d,]+\.\d{2})$', l)
        items.append(dict(line=int(m[1]), description=m[2], quantity=int(m[3]), unit_price=num(m[4]), line_total=num(m[5])))
    head = dict(
        supplier=find(r'^INVOICE\n(.+)$', t), customer=find(r'^(.+?) Invoice No:', t),
        invoice_number=find(r'Invoice No: (\S+)', t), invoice_date=find(r'Invoice date: (\S+)', t, iso),
        due_date=find(r'Due date: (\S+)', t, iso), customer_reference=find(r'Customer PO: (.+)$', t),
        payment_terms=find(r'Payment terms: (.+)$', t), currency='MUR',
        subtotal=find(r'^Subtotal (\S+)', t, num), tax_rate=find(r'^VAT (\d+)%', t, float),
        tax_amount=find(r'^VAT \d+% (\S+)', t, num), total=find(r'^TOTAL DUE \(Rs\) (\S+)', t, num))
    return head, items


def parse_coral(t):
    items = []
    for n, m in enumerate(re.finditer(r'^(CT\d+) (.+) (\d+) ([\d,]+\.\d{2})(?: (\d+)%)? ([\d,]+\.\d{2})$', t, re.M), 1):
        items.append(dict(line=n, item_code=m[1], description=m[2], quantity=int(m[3]), unit_price=num(m[4]),
                          discount_percent=float(m[5]) if m[5] else 0.0, line_total=num(m[6])))
    head = dict(
        supplier='Coral Tech ' + find(r'^(Distribution Ltd)', t), customer=find(r'^(.+?) DELIVERED TO', t),
        invoice_number=find(r'No\. (CT\S+ \S+)', t), invoice_date=find(r'Date: (\S+)', t, iso),
        customer_reference=find(r'Order ref: (.+)$', t), payment_terms=find(r'Terms: (.+)$', t), currency='MUR',
        subtotal=find(r'^Subtotal (\S+)', t, num), delivery=find(r'^Delivery (\S+)', t, num),
        tax_rate=find(r'^VAT (\d+)%', t, float), tax_amount=find(r'^VAT \d+% (\S+)', t, num),
        total=find(r'^AMOUNT DUE \(Rs\) (\S+)', t, num))
    return head, items


def parse_maison(t):
    amount = r'(\d{1,3}(?: \d{3})*,\d{2})'
    items = []
    for n, l in enumerate(section(t, 'Désignation', 'Total hors taxes'), 1):
        m = re.match(rf'^(.+?) (\d+) {amount} {amount}$', l)
        items.append(dict(line=n, description=m[1], quantity=int(m[2]), unit_price=num(m[3]), line_total=num(m[4])))
    head = dict(
        supplier=t.splitlines()[0], customer=find(r'^Date : \S+ (.+)$', t),
        invoice_number=find(r'Facture n° (\S+)', t), invoice_date=find(r'^Date : (\S+)', t, iso),
        due_date=find(r'Échéance : (\S+)', t, iso), event_date=find(r"Date de l'événement : (\S+)", t, iso),
        currency='MUR', subtotal=find(rf'^Total hors taxes {amount}', t, num),
        discount=find(rf'^Remise .* moins {amount}', t, num), tax_rate=find(r'^TVA (\d+) %', t, float),
        tax_amount=find(rf'^TVA \d+ % {amount}', t, num), total=find(rf'^Total à payer \(Rs\) {amount}', t, num),
        deposit=find(rf'Acompte de Rs {amount}', t, num), balance_due=find(rf'Solde restant à payer : Rs {amount}', t, num))
    return head, items


def parse_hlp(t):
    items = []
    for n, l in enumerate(section(t, 'DESCRIPTION AMOUNT', 'SUBTOTAL'), 1):
        m = re.match(r'^(.+) \$ ([\d ]+,\d{2})$', l)
        items.append(dict(line=n, description=m[1], line_total=num(m[2])))
    head = dict(
        supplier=find(r'^INVOICE\n(.+)$', t), customer=find(r'BILL TO: .+\n\S+@\S+ (.+)$', t),
        invoice_number=find(r'INVOICE # (\S+)', t), invoice_date=find(r'DATE: (.+)$', t, lambda s: iso(s, '%B %d, %Y')),
        currency='USD', subtotal=find(r'^SUBTOTAL \$ (.+)$', t, num),
        tax_rate=find(r'^TAX RATE ([\d,]+)%', t, num), tax_amount=find(r'^SALES TAX \$ (.+)$', t, num),
        delivery=find(r'^OTHER \$ (.+)$', t, num), total=find(r'^TOTAL \$ (.+)$', t, num))
    return head, items


def parse_shadow(t):
    items = []
    for n, l in enumerate(section(t, 'QTY ITEM #', 'TOTAL DISCOUNT'), 1):
        m = re.match(r'^(\d+) (\d+) (.+) \$([\d,.]+) \$([\d,.]+) \$([\d,.]+)$', l)
        items.append(dict(line=n, item_code=m[2], description=m[3], quantity=int(m[1]), unit_price=num(m[4]),
                          discount_amount=num(m[5]), line_total=num(m[6])))
    terms = re.search(r'^\S+ \S+ Sales (.+) (\d{2}/\d{2}/\d{4}) (.+) (\d{2}/\d{2}/\d{4})$', t, re.M)
    head = dict(
        supplier=find(r'^(SHADOW BEAUTY)', t, str.title), customer=find(r'^TO .+\n(.+)$', t),
        invoice_number=find(r'INVOICE # (\S+)', t), invoice_date=find(r'DATE (\S+)', t, iso),
        due_date=iso(terms[4]), customer_reference=find(r'CUSTOMER ID (\S+)', t),
        payment_terms=terms[3], currency='USD',
        subtotal=find(r'^SUBTOTAL \$(\S+)', t, num), tax_amount=find(r'^SALES TAX \$(\S+)', t, num),
        total=find(r'^TOTAL \$(\S+)', t, num))
    return head, items


LAYOUTS = {
    '# Description Qty': parse_bluewave,
    'Item Description Qty Unit Rs Disc %': parse_coral,
    'Désignation Qté': parse_maison,
    'DESCRIPTION AMOUNT': parse_hlp,
    'QTY ITEM # DESCRIPTION': parse_shadow,
}

## Extraction and checks

In [4]:
INVOICE_COLUMNS = ['source_file', 'supplier', 'customer', 'invoice_number', 'invoice_date', 'due_date', 'event_date',
                   'customer_reference', 'payment_terms', 'currency', 'subtotal', 'discount', 'delivery', 'tax_rate',
                   'tax_amount', 'total', 'deposit', 'balance_due', 'line_count', 'lines_match_subtotal', 'total_reconciles']
ITEM_COLUMNS = ['source_file', 'invoice_number', 'line', 'item_code', 'description', 'quantity', 'unit_price',
                'discount_percent', 'discount_amount', 'line_total']

invoices, items = [], []
for pdf in sorted(PDF_DIR.glob('*.pdf')):
    with pdfplumber.open(pdf) as doc:
        text = '\n'.join(p.extract_text() for p in doc.pages)
    parser = next(f for key, f in LAYOUTS.items() if key in text)
    head, lines = parser(text)
    head = {'source_file': pdf.name, 'discount': 0.0, 'delivery': 0.0, 'tax_rate': 0.0,
            'deposit': 0.0, **{k: v for k, v in head.items() if v is not None}}
    head.setdefault('balance_due', head['total'] - head['deposit'])
    line_sum = round(sum(i['line_total'] for i in lines), 2)
    head['line_count'] = len(lines)
    head['lines_match_subtotal'] = abs(line_sum - head['subtotal']) < 0.01
    expected = head['subtotal'] - head['discount'] + head['delivery'] + head['tax_amount']
    head['total_reconciles'] = abs(expected - head['total']) < 0.01
    invoices.append(head)
    items += [{'source_file': pdf.name, 'invoice_number': head['invoice_number'], **i} for i in lines]

invoices = pd.DataFrame(invoices).reindex(columns=INVOICE_COLUMNS)
items = pd.DataFrame(items).reindex(columns=ITEM_COLUMNS)

with pd.ExcelWriter(OUT / 'Task4.xlsx', engine='openpyxl') as xw:
    invoices.to_excel(xw, sheet_name='Invoices', index=False)
    items.to_excel(xw, sheet_name='Line items', index=False)
    for ws in xw.sheets.values():
        ws.freeze_panes = 'A2'
        for c in ws.columns:
            ws.column_dimensions[c[0].column_letter].width = min(45, max(12, max(len(str(x.value or '')) for x in c) + 2))

invoices

,source_file,supplier,customer,invoice_number,invoice_date,due_date,event_date,customer_reference,payment_terms,currency,...,discount,delivery,tax_rate,tax_amount,total,deposit,balance_due,line_count,lines_match_subtotal,total_reconciles
0,bluewave-invoice.pdf,Bluewave Stationery Ltd,Horizon Dental Clinic,BW/2026/0412,2026-09-03,2026-10-03,NaN,HDC 7781,30 days,MUR,...,0.0,0.0,15.0,3112.50,23862.50,0.0,23862.50,6,True,True
1,coral_tech-invoice.pdf,Coral Tech Distribution Ltd,Lumina Learning Centre,CT26 003581,2026-09-28,NaN,NaN,LLC 2209,Net 14 days,MUR,...,0.0,450.0,15.0,55907.62,428625.12,0.0,428625.12,32,True,True
2,hlp-invoice.pdf,HLP Manufacturing INC,Robotine Manufacturing,100,2026-10-19,NaN,NaN,NaN,NaN,USD,...,0.0,0.0,0.0,0.00,11250.00,0.0,11250.00,5,True,True
3,maison_vanille-invoice.pdf,Maison Vanille Traiteur,Lagune Events Ltée,2026/087,2026-09-12,2026-10-12,2026-09-05,NaN,NaN,MUR,...,1600.0,0.0,15.0,4560.00,34960.00,10000.0,24960.00,5,True,True
4,shadow-invoice.pdf,Shadow Beauty,Central Beauty,54167,2026-10-19,2026-11-18,NaN,BK06948,On Delivery,USD,...,0.0,0.0,0.0,0.00,313.90,0.0,313.90,4,True,True


## Line items

In [5]:
items

,source_file,invoice_number,line,item_code,description,quantity,unit_price,discount_percent,discount_amount,line_total
0,bluewave-invoice.pdf,BW/2026/0412,1,NaN,"Printer paper A4 80gsm, box of 5 reams",12.0,685.00,NaN,NaN,8220.00
1,bluewave-invoice.pdf,BW/2026/0412,2,NaN,"Ballpoint pens blue, box of 50",8.0,240.00,NaN,NaN,1920.00
2,bluewave-invoice.pdf,BW/2026/0412,3,NaN,Stapler heavy duty,4.0,520.00,NaN,NaN,2080.00
3,bluewave-invoice.pdf,BW/2026/0412,4,NaN,Lever arch files,30.0,95.00,NaN,NaN,2850.00
4,bluewave-invoice.pdf,BW/2026/0412,5,NaN,"Whiteboard markers, pack of 12",10.0,310.00,NaN,NaN,3100.00
5,bluewave-invoice.pdf,BW/2026/0412,6,NaN,Desk organiser,6.0,430.00,NaN,NaN,2580.00
6,coral_tech-invoice.pdf,CT26 003581,1,CT1000,USB C cable 1m,3.0,180.00,0.0,NaN,540.00
7,coral_tech-invoice.pdf,CT26 003581,2,CT1007,Wireless mouse,5.0,650.00,0.0,NaN,3250.00
8,coral_tech-invoice.pdf,CT26 003581,3,CT1014,Mechanical keyboard,1.0,2450.00,10.0,NaN,2205.00
9,coral_tech-invoice.pdf,CT26 003581,4,CT1021,HDMI cable 2m,1.0,320.00,0.0,NaN,320.00
